<a href="https://colab.research.google.com/github/treborskrub/Modular-/blob/main/ModularUnivContEngine_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# =====================================================
# UNIVERSAL CONTRACTION ENGINE - SINGLE FILE VERSION
# All modules are now one clean script with full type hints
# =====================================================

from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any
import math
import re


# =====================================================
# 1. CORE KERNEL (immutable mathematical truth source)
# =====================================================
def test_core_equation_trig(epsilon: float, phi: float) -> float:
    """Immutable mathematical kernel — the ONLY truth source."""
    pi_trig = 4.0 * math.atan(1.0)
    term_expansion = (1.0 / 3.0) + epsilon
    term_contraction = (1.0 / 3.0) - epsilon

    step_1 = term_expansion * pi_trig
    step_2 = step_1 * phi
    final_output = step_2 * term_contraction

    checksum = ((1.0 / 9.0) - (epsilon ** 2)) * pi_trig * phi
    if abs(final_output - checksum) >= 1e-15:
        raise ValueError("Core kernel integrity failure — precision drift detected!")

    return final_output


# =====================================================
# 2. LINGUISTIC MAPPER
# =====================================================
@dataclass
class EvidenceQuantum:
    """Atomic unit of grounding evidence."""
    claim_fragment: str
    support_score: float          # 0.0 to 1.0
    confidence: float             # source reliability

class LinguisticMapper:
    def __init__(self, phi_base: float = (1 + 5**0.5) / 2) -> None:
        self.phi_base = phi_base

    def extract_claims(self, text: str) -> List[str]:
        """Splits full text into individual auditable claims."""
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        return [s.strip() for s in sentences if len(s.strip()) > 10]

    def measure_assertion_confidence(self, claim: str) -> float:
        """Determines assertion strength via marker words."""
        words = claim.lower().split()
        strong_markers = {"is", "are", "always", "never", "definitely", "certainly", "proven", "fact"}
        weak_markers = {"might", "could", "perhaps", "maybe", "typically", "usually"}

        strong_count = sum(1 for w in words if w in strong_markers)
        weak_count = sum(1 for w in words if w in weak_markers)

        base = 0.5 + (strong_count * 0.08) - (weak_count * 0.06)
        return min(1.0, max(0.1, base))

    def accumulate_grounding(self, evidence: Optional[List[EvidenceQuantum]]) -> float:
        """Scales evidence using geometric decay."""
        if not evidence:
            return 0.0
        total_support = 0.0
        for idx, eq in enumerate(evidence):
            weight = self.phi_base ** (-(idx + 1))
            total_support += eq.support_score * eq.confidence * (1 - weight)
        return min(1.0, total_support / self.phi_base)


# =====================================================
# 3. AUDIT PROCESSOR (your Universal Process Engine)
# =====================================================
@dataclass
class Step:
    """Single auditable step in the pipeline."""
    step_id: int
    confidence: float
    grounding: float
    note: str = ""

@dataclass
class Result:
    """Audit result with full contraction history."""
    round_id: int
    step_id: int
    original_confidence: float
    original_grounding: float
    adjusted_confidence: float
    adjusted_grounding: float
    shortfall: float
    lambda_ratio: float
    gate_fired: bool
    verdict: str
    correction: float
    note: str

class UniversalProcessEngine:
    """Universal recursive contraction + audit engine."""
    def __init__(self, threshold: float = 1/3, anomaly_floor: float = 0.0001) -> None:
        self.threshold = threshold
        self.anomaly_floor = anomaly_floor
        self.initial_shortfall: Optional[float] = None
        self.history: List[Result] = []

    def apply_anomaly_floor(self, confidence: float, grounding: float) -> tuple[float, float]:
        confidence = max(0.0, confidence - self.anomaly_floor)
        grounding = max(0.0, grounding + self.anomaly_floor)
        return confidence, grounding

    def compute_shortfall(self, confidence: float, grounding: float) -> float:
        return max(0.0, confidence - grounding)

    def compute_lambda(self, shortfall: float) -> float:
        if self.initial_shortfall is None or self.initial_shortfall == 0:
            return 1.0
        return shortfall / self.initial_shortfall

    def classify(self, shortfall: float, lambda_ratio: float) -> str:
        if shortfall <= self.threshold:
            return "PASS"
        if lambda_ratio < 1.0:
            return "WARN"
        return "FAIL"

    def correction(self, shortfall: float, lambda_ratio: float) -> float:
        if shortfall <= self.threshold:
            return 0.0
        base = shortfall - self.threshold
        if lambda_ratio < 1.0:
            return min(1.0, base * 1.5)
        return min(1.0, base + self.anomaly_floor)

    def revise_step(self, step: Step, corr: float) -> Step:
        new_conf = max(0.0, step.confidence - corr * 0.4)
        new_ground = min(1.0, step.grounding + corr * 0.6)
        return Step(
            step_id=step.step_id,
            confidence=new_conf,
            grounding=new_ground,
            note=step.note + " | revised"
        )

    def audit_step(self, step: Step, round_id: int) -> Result:
        adj_conf, adj_ground = self.apply_anomaly_floor(step.confidence, step.grounding)
        shortfall = self.compute_shortfall(adj_conf, adj_ground)

        if self.initial_shortfall is None:
            self.initial_shortfall = shortfall

        lambda_ratio = self.compute_lambda(shortfall)
        gate_fired = shortfall > self.threshold
        verdict = self.classify(shortfall, lambda_ratio)
        corr = self.correction(shortfall, lambda_ratio)

        result = Result(
            round_id=round_id, step_id=step.step_id,
            original_confidence=step.confidence, original_grounding=step.grounding,
            adjusted_confidence=adj_conf, adjusted_grounding=adj_ground,
            shortfall=shortfall, lambda_ratio=lambda_ratio, gate_fired=gate_fired,
            verdict=verdict, correction=corr, note=step.note
        )
        self.history.append(result)
        return result

    def run(self, timeline: List[Step], max_rounds: int = 5) -> List[Result]:
        current = timeline
        for round_id in range(1, max_rounds + 1):
            next_timeline: List[Step] = []
            any_fail = False
            for step in current:
                result = self.audit_step(step, round_id)
                if result.verdict == "FAIL":
                    any_fail = True
                next_timeline.append(self.revise_step(step, result.correction))
            if not any_fail:
                break
            current = next_timeline
        return self.history

    def export_json(self) -> str:
        return str([asdict(r) for r in self.history])


# =====================================================
# 4. GATE SYNC (asymmetric handshake)
# =====================================================
from typing import Dict

class UniversalGateSync:
    """Asymmetric token handshake engine."""
    def __init__(self, target_steps: int = 3) -> None:
        self.target_steps = target_steps

    def process_handshake_step(self, token_packet: Dict[str, str], current_step: int) -> str:
        if current_step == 1 and token_packet.get("cmd") == "PROPOSAL":
            return "ACK_TOKEN_GENERATED"
        elif current_step == 2 and token_packet.get("cmd") == "ACKNOWLEDGMENT":
            return "AUTH_TOKEN_GENERATED"
        elif current_step == self.target_steps and token_packet.get("cmd") == "AUTHORIZATION":
            return "PIPELINE_UNLOCKED"
        return "CONNECTION_TERMINATED"


# =====================================================
# 5. SIEVE PROCESSOR
# =====================================================
class UniversalSieveEngine:
    """Universal sieve engine for perplexity-based verification."""
    def __init__(self, expected_threshold: float = 0.5) -> None:
        self.expected_threshold = expected_threshold

    def run_sieve(self, packet: Dict[str, Any], pass_count: int) -> Dict[str, Any]:
        current_perplexity = packet.get("base_perplexity", 1.0) * (1.0 - packet.get("current_token_loss", 0.05) * pass_count)

        if current_perplexity < self.expected_threshold:
            return {"status": "VERIFIED", "meta": {"final_perplexity": current_perplexity}}
        elif pass_count >= 5:
            return {"status": "DIVERGED_FALLBACK", "reason": "Perplexity too high after multiple passes"}
        else:
            return {"status": "PROCESSING", "lambda": current_perplexity}


# =====================================================
# 6. PIPELINE LINKER (orchestrator)
# =====================================================
class UPCPipelineLinker:
    """Full orchestrator — glues gate + sieve + your audit engine."""
    def __init__(self, gate_engine: Any, sieve_engine: Any) -> None:
        self.gate = gate_engine
        self.sieve = sieve_engine
        self.active_pipelines: Dict[str, Dict[str, Any]] = {}

    def ingest_packet(self, session_id: str, packet: Dict[str, Any]) -> Dict[str, Any]:
        state = self.active_pipelines.get(session_id, {"phase": "GATE", "step": 1})

        if state["phase"] == "GATE":
            gate_status = self.gate.process_handshake_step(packet, state["step"])
            if gate_status == "CONNECTION_TERMINATED":
                self.active_pipelines.pop(session_id, None)
                return {"status": "HALTED", "reason": "Handshake Authentication Failure"}
            elif gate_status == "PIPELINE_UNLOCKED":
                state["phase"] = "SIEV"
                state["pass_count"] = 1
                self.active_pipelines[session_id] = state
                return {"status": "PIPELINE_UNLOCKED", "phase": "SIEV", "next_expected_pass": 1}
            else:
                state["step"] += 1
                self.active_pipelines[session_id] = state
                return {"status": "HANDSHAKING", "next_expected_step": state["step"]}

        elif state["phase"] == "SIEV":
            packet["base_perplexity"] = packet.get("initial_delta", 1.0)
            packet["current_token_loss"] = packet.get("current_delta", 0.05)
            sieve_result = self.sieve.run_sieve(packet, state["pass_count"])

            if sieve_result["status"] == "PROCESSING":
                state["pass_count"] += 1
                self.active_pipelines[session_id] = state
                return {"status": "PROCESSING_STREAM", "lambda": sieve_result["lambda"]}
            elif sieve_result["status"] == "VERIFIED":
                self.active_pipelines.pop(session_id, None)
                return {"status": "SUCCESS_STREAM_VERIFIED", "meta": sieve_result["meta"]}
            elif sieve_result["status"] == "DIVERGED_FALLBACK":
                self.active_pipelines.pop(session_id, None)
                return {"status": "CRITICAL_DIVERGENCE_EVACUATED"}

        return {"status": "UNKNOWN_ERROR"}


# =====================================================
# EXAMPLE USAGE (copy this block to test)
# =====================================================
if __name__ == "__main__":
    print("🚀 Universal Contraction Engine — Single-File Mode")
    print("=" * 60)

    # 1. Your audit engine
    engine = UniversalProcessEngine(threshold=1/3)

    # 2. Wrap for GateSync compatibility
    class YourEngineForGate(UniversalProcessEngine):
        def process_handshake_step(self, token_packet: Dict[str, str], current_step: int) -> str:
            if current_step == 1:
                result = self.audit_step(Step(1, 0.8, 0.7), 1)
                return "ACK_TOKEN_GENERATED" if result.verdict == "PASS" else "CONNECTION_TERMINATED"
            return "CONNECTION_TERMINATED"

    gate = YourEngineForGate()
    sieve = UniversalSieveEngine(expected_threshold=0.5)
    linker = UPCPipelineLinker(gate, sieve)

    # Run a live pipeline
    sample_packet = {"cmd": "PROPOSAL", "claim": "The golden ratio is undeniably verified by structural mathematical bounds."}
    session_id = "session_123"

    print(f"Ingesting packet for {session_id}...")
    response = linker.ingest_packet(session_id, sample_packet)
    print("Pipeline Response:", response)

    print("\n✅ System ready. All modules are modular, type-hinted, and production-ready.")

🚀 Universal Contraction Engine — Single-File Mode
Ingesting packet for session_123...
Pipeline Response: {'status': 'HANDSHAKING', 'next_expected_step': 2}

✅ System ready. All modules are modular, type-hinted, and production-ready.
